In [1]:
import asyncio
import os
import faiss
from datetime import datetime

# 💥 IMPORT CHUẨN THEO TÀI LIỆU CẬU GỬI:
from llama_index.core import Document, VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore

# Khởi tạo dữ liệu cấu trúc để chạy mượt mà trong môi trường Async của Jupyter
import nest_asyncio
nest_asyncio.apply()

# Thư mục lưu file index cứng
PERSIST_DIR = "./storage_test_faiss"

# =============================================================================
# 0. KHỞI TẠO HẠ TẦNG CHẠY OFFLINE (FAST-EMBED & FAISS)
# =============================================================================
print("⏳ Bước 0: Đang kích hoạt FastEmbed. Hệ thống sẽ tự động cấu hình mô hình...")
embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")

print("⏳ Đang băm chữ và nạp vào kho FAISS trên RAM...")
dataset = [
    Document(text="Quy định nghỉ phép: Mỗi năm nhân viên có 12 ngày phép hưởng nguyên lương."),
    Document(text="Chính sách thưởng: Thưởng tháng 13 được chi trả vào kỳ lương cuối cùng của tháng 1 dương lịch."),
    Document(text="Bảo mật source code: Tuyệt đối không được public mã nguồn của công ty lên github cá nhân.")
]

d = 384  
faiss_index = faiss.IndexFlatL2(d)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Tạo index trên RAM
index = VectorStoreIndex.from_documents(dataset, storage_context=storage_context, embed_model=embed_model)

# 🔥 ĐÚNG THEO TÀI LIỆU: Lưu đống RAM này xuống ổ cứng thành file vật lý
print(f"⏳ Đang Persist dữ liệu xuống thư mục: {PERSIST_DIR}")
index.storage_context.persist(persist_dir=PERSIST_DIR)
print("✅ Đã ghi file thành công!")

# Giả lập một Semantic Cache L2
semantic_cache = {
    "công ty có bao nhiêu ngày phép": "Theo quy định, bạn có tối đa 12 ngày phép hưởng lương mỗi năm.",
}

# =============================================================================
# 1. CÁC HÀM LOGIC KIỂM TRA
# =============================================================================
async def input_guard(query: str):
    if "hack" in query.lower():
        return False, "⚠️ HỆ THỐNG CHẶN: Phát hiện từ khóa nguy hiểm!"
    return True, query

async def heuristic_router(query: str):
    if any(w in query.lower() for w in ["nghỉ phép", "thưởng", "source code"]):
        return "RAG_KNOWLEDGE"
    return "GENERAL_LLM"

# =============================================================================
# 2. ĐIỀU PHỐI LUỒNG WORKFLOW CHUẨN HỘI ĐỒNG (LOAD TỪ STORAGE)
# =============================================================================
async def run_optimized_workflow(user_query: str):
    start_time = datetime.now()
    print(f"\n📥 USER HỎI: '{user_query}'")
    
    # Vòng gửi xe
    is_safe, clean_query = await input_guard(user_query)
    if not is_safe: return clean_query

    # Router
    route = await heuristic_router(clean_query)
    print(f"  ↳ [Router] Nhận diện luồng: {route}")

    # Nếu rơi vào luồng RAG -> Đọc file FAISS lên RAM để quét
    if route == "RAG_KNOWLEDGE":
        print(f"  ↳ [Cache Miss] Đang LOAD file index từ ổ cứng lên RAM...")
        
        # 🔥 ĐÚNG THEO TÀI LIỆU CẬU GỬI: Re-create storage context từ thư mục persist
        # Ở đây ta map đúng kho chứa vector_store là FAISS
        new_storage_context = StorageContext.from_defaults(
            persist_dir=PERSIST_DIR, 
            vector_store=vector_store
        )
        
        # 🔥 ĐÚNG THEO TÀI LIỆU CẬU GỬI: Dùng load_index_from_storage sạch sẽ
        new_index = load_index_from_storage(new_storage_context, embed_model=embed_model)
        
        # Quét tri thức
        retriever = new_index.as_retriever(similarity_top_k=1)
        nodes = retriever.retrieve(clean_query)
        
        context = nodes[0].text
        print(f"  ↳ [FAISS Hit] Đã bốc được tài liệu: '{context[:45]}...'")
        print("  🤖 [LLM Call] Đang đẩy context vào LLM để sinh câu trả lời...")
        
        elapsed = (datetime.now() - start_time).total_seconds()
        return f"  ✨ [RESULT]: Dựa vào tài liệu nội bộ: {context} (Tốc độ: {elapsed:.4f}s)"
    
    return "  🤖 Luồng chat thông thường."

# Chạy thử kịch bản đọc file thực tế
async def main():
    print("="*60)
    print(await run_optimized_workflow("Tiền thưởng tháng 13 tính thế nào?"))

await main()

⏳ Bước 0: Đang kích hoạt FastEmbed. Hệ thống sẽ tự động cấu hình mô hình...


/home/duoc/app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-28 11:07:05.997920760 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


⏳ Đang băm chữ và nạp vào kho FAISS trên RAM...
⏳ Đang Persist dữ liệu xuống thư mục: ./storage_test_faiss
✅ Đã ghi file thành công!

📥 USER HỎI: 'Tiền thưởng tháng 13 tính thế nào?'
  ↳ [Router] Nhận diện luồng: RAG_KNOWLEDGE
  ↳ [Cache Miss] Đang LOAD file index từ ổ cứng lên RAM...
  ↳ [FAISS Hit] Đã bốc được tài liệu: 'Chính sách thưởng: Thưởng tháng 13 được chi t...'
  🤖 [LLM Call] Đang đẩy context vào LLM để sinh câu trả lời...
  ✨ [RESULT]: Dựa vào tài liệu nội bộ: Chính sách thưởng: Thưởng tháng 13 được chi trả vào kỳ lương cuối cùng của tháng 1 dương lịch. (Tốc độ: 0.0077s)


In [1]:
"""RAG ZERO v2.1 — compact single-file RAG with BM25 + FAISS + RRF fusion."""

import asyncio, hashlib, logging, unicodedata, tempfile, os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Set
import json
import faiss
import numpy as np
from fastembed import TextEmbedding
from rank_bm25 import BM25Okapi

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("rag")

PERSIST_DIR    = Path("./rag_storage")
EMBED_MODEL    = "BAAI/bge-small-en-v1.5"
DIM            = 384
CHUNK_SIZE     = 512
CHUNK_OVERLAP  = 50
RRF_K          = 60
RRF_GAP        = 0.6
BM25_MIN_RATIO = 0.3
L2_THRESHOLD   = 0.85
L1_MAX         = 1000


# ── Async embed helper ────────────────────────────────────────────────────────

async def _aembed(embed_model: TextEmbedding, text: str) -> np.ndarray:
    """
    fastembed.TextEmbedding chỉ có embed() đồng bộ (sync).
    Bọc trong run_in_executor để không block event loop.
    embed() trả về generator → list()[0] để lấy vector.
    """
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(
        None, lambda: list(embed_model.embed([text]))[0]
    )


# ── Data classes ──────────────────────────────────────────────────────────────

@dataclass
class Doc:
    text: str
    metadata: dict = field(default_factory=dict)

@dataclass
class Chunk:
    score: float
    text:  str
    meta:  dict

@dataclass
class Result:
    query:  str
    chunks: List[Chunk]
    source: str


# ── Synonyms ──────────────────────────────────────────────────────────────────

SYNONYMS = {
    "thưởng tháng 13": ["tiền thưởng cuối năm", "bonus tết", "lương tháng 13", "thưởng tết"],
    "nghỉ phép năm":   ["annual leave", "phép năm", "ngày phép hưởng lương"],
    "nghỉ phép":       ["ngày phép", "leave", "day off", "xin nghỉ"],
    "public code":     ["public mã nguồn", "đăng github", "upload source", "chia sẻ code"],
    "lương":           ["tiền công", "thu nhập"],
    "bảo mật":         ["security", "an ninh", "confidential"],
}

_reverse = {v.lower(): k for k, vs in SYNONYMS.items() for v in vs}
_reverse.update({k.lower(): k for k in SYNONYMS})
_reverse = dict(sorted(_reverse.items(), key=lambda x: len(x[0]), reverse=True))

def _no_accent(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def expand_query(text: str) -> List[str]:
    result = text.lower()
    for variant, canonical in _reverse.items():
        if variant in result and variant != canonical:
            result = result.replace(variant, canonical)
    variants = list({text, result, _no_accent(result)})
    return variants


# ── Cache (L1 exact + L2 semantic) ───────────────────────────────────────────

class Cache:
    def __init__(self, embed: TextEmbedding):
        self._l1: Dict[str, Chunk] = {}
        self._embed = embed
        self._idx   = faiss.IndexFlatIP(DIM)
        self._items: List[Chunk] = []

    def get_l1(self, q: str) -> Optional[Chunk]:
        return self._l1.get(q.lower().strip())

    def put_l1(self, q: str, c: Chunk):
        if len(self._l1) >= L1_MAX:
            del self._l1[next(iter(self._l1))]
        self._l1[q.lower().strip()] = c

    async def get_l2(self, q: str) -> Optional[Chunk]:
        if not self._idx.ntotal:
            return None
        v = _vec(await _aembed(self._embed, q))  # ← fixed
        scores, ids = self._idx.search(v, 1)
        return self._items[ids[0][0]] if scores[0][0] >= L2_THRESHOLD else None

    async def put_l2(self, q: str, c: Chunk):
        self._idx.add(_vec(await _aembed(self._embed, q)))  # ← fixed
        self._items.append(c)

    def stats(self): return {"l1": len(self._l1), "l2": self._idx.ntotal}


# ── Helpers ───────────────────────────────────────────────────────────────────

def _vec(emb) -> np.ndarray:
    v = np.array([emb], dtype=np.float32)
    faiss.normalize_L2(v)
    return v

def _key(text: str) -> str:
    return hashlib.md5(text.encode()).hexdigest()[:16]

def _rrf(lists: List[List[Chunk]], k: int = RRF_K) -> List[Chunk]:
    ranks: Dict[int, Dict] = {}
    for list_idx, lst in enumerate(lists):
        for rank, c in enumerate(lst):
            key = hash(c.text)
            if key not in ranks:
                ranks[key] = {"chunk": c, "score": 0.0, "lists": set()}
            ranks[key]["score"] += 1.0 / (k + rank + 1)
            ranks[key]["lists"].add(list_idx)
    sorted_items = sorted(ranks.values(), key=lambda x: x["score"], reverse=True)
    if not sorted_items:
        return []
    winner_score = sorted_items[0]["score"]
    n_lists = len(lists)
    return [
        Chunk(score=v["score"], text=v["chunk"].text, meta=v["chunk"].meta)
        for v in sorted_items
        if v["score"] >= winner_score * RRF_GAP or len(v["lists"]) >= n_lists
    ]


# ── Store (ingest + search) ───────────────────────────────────────────────────

class Store:
    def __init__(self, embed: TextEmbedding):
        self._embed  = embed
        self._idx    = faiss.IndexFlatIP(DIM)
        self._texts: List[str]       = []
        self._metas: List[dict]      = []
        self._bm25_corpus: List[List[str]] = []
        self._bm25: Optional[BM25Okapi]    = None
        self._hashes: Set[str]             = set()
        PERSIST_DIR.mkdir(exist_ok=True)
        self._load()

    # ── Ingest ────────────────────────────────────────────────────────────────

    async def add(self, doc: Doc) -> str:
        text = doc.text.strip()
        h    = hashlib.sha256(text.encode()).hexdigest()[:16]
        if not text or h in self._hashes:
            return "duplicate" if text else "skipped"

        chunks = self._chunk(text)
        meta   = {"source_id": doc.metadata.get("source_id", h), **doc.metadata}

        # PARALLEL EMBEDDING: gather các coroutine _aembed concurrently
        embs_list = await asyncio.gather(*[_aembed(self._embed, c) for c in chunks])  # ← fixed
        embs = np.array(embs_list, dtype=np.float32)
        faiss.normalize_L2(embs)
        self._idx.add(embs)

        for i, c in enumerate(chunks):
            self._texts.append(c)
            self._metas.append({**meta, "chunk_idx": i})
            self._bm25_corpus.append(c.lower().split())

        self._bm25 = BM25Okapi(self._bm25_corpus)
        self._hashes.add(h)
        self._save()
        log.info(f"[add] {h} → {len(chunks)} chunk(s)")
        return "ok"

    def _chunk(self, text: str) -> List[str]:
        words = text.split()
        if len(words) <= CHUNK_SIZE:
            return [text]
        out, start = [], 0
        while start < len(words):
            out.append(" ".join(words[start:start + CHUNK_SIZE]))
            start += CHUNK_SIZE - CHUNK_OVERLAP
        return out

    # ── Search ────────────────────────────────────────────────────────────────

    async def vector(self, query: str, k: int) -> List[Chunk]:
        if not self._idx.ntotal:
            return []
        scores, ids = self._idx.search(_vec(await _aembed(self._embed, query)), k)  # ← fixed
        return [
            Chunk(float(s), self._texts[i], self._metas[i])
            for s, i in zip(scores[0], ids[0]) if i >= 0 and s > 0
        ]

    def bm25(self, queries: List[str], k: int) -> List[Chunk]:
        if not self._bm25:
            return []
        raw = np.max(
            [self._bm25.get_scores(q.lower().split()) for q in queries], axis=0
        )
        top  = np.argsort(raw)[::-1][:k]
        mx   = raw.max() or 1
        best = raw[top[0]] if len(top) else 1
        return [
            Chunk(float(raw[i] / mx), self._texts[i], self._metas[i])
            for i in top if raw[i] > 0 and raw[i] >= best * BM25_MIN_RATIO
        ]

    # ── Persist ───────────────────────────────────────────────────────────────

    def _save(self):
        """Atomic write: ghi ra temp rồi rename, tránh corrupted file nếu crash giữa chừng."""
        data = {
            "texts":  self._texts,
            "metas":  self._metas,
            "corpus": self._bm25_corpus,
            "hashes": list(self._hashes),
        }
        json_path = PERSIST_DIR / "data.json"
        tmp_json = tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=PERSIST_DIR, delete=False, suffix=".tmp"
        )
        try:
            json.dump(data, tmp_json, ensure_ascii=False)
            tmp_json.close()
            os.replace(tmp_json.name, json_path)
        except Exception:
            os.unlink(tmp_json.name)
            raise

        index_path = PERSIST_DIR / "faiss.index"
        tmp_index = tempfile.NamedTemporaryFile(dir=PERSIST_DIR, delete=False, suffix=".tmp")
        try:
            tmp_index.close()
            faiss.write_index(self._idx, tmp_index.name)
            os.replace(tmp_index.name, index_path)
        except Exception:
            if os.path.exists(tmp_index.name):
                os.unlink(tmp_index.name)
            raise

    def _load(self):
        """Graceful degradation: nếu file hỏng, log warning và khởi động store rỗng."""
        dp, fp = PERSIST_DIR / "data.json", PERSIST_DIR / "faiss.index"
        if not (dp.exists() and fp.exists()):
            return
        try:
            with open(dp, "r", encoding="utf-8") as f:
                d = json.load(f)
            self._texts, self._metas = d["texts"], d["metas"]
            self._bm25_corpus = d["corpus"]
            self._bm25 = BM25Okapi(self._bm25_corpus) if self._bm25_corpus else None
            self._hashes = set(d["hashes"])
            self._idx = faiss.read_index(str(fp))
            log.info(f"[load] {len(self._texts)} chunks restored")
        except Exception as e:
            log.warning(f"[load] Corrupted persist files, starting fresh: {e}")
            self._texts, self._metas, self._bm25_corpus = [], [], []
            self._bm25, self._hashes = None, set()
            self._idx = faiss.IndexFlatIP(DIM)


# ── RAG Pipeline ──────────────────────────────────────────────────────────────

class RAG:
    def __init__(self):
        self._embed = TextEmbedding(model_name=EMBED_MODEL)
        self._store = Store(self._embed)
        self._cache = Cache(self._embed)

    async def add(self, text: str, **meta) -> str:
        return await self._store.add(Doc(text, meta))

    async def search(self, query: str, top_k: int = 3) -> Result:
        query = query.strip()

        if c := self._cache.get_l1(query):
            return Result(query, [c], "cache_l1")
        if c := await self._cache.get_l2(query):
            return Result(query, [c], "cache_l2")

        variants  = expand_query(query)
        bm25_hits = self._store.bm25(variants, top_k * 3)
        vec_hits  = await self._store.vector(variants[0], top_k * 3)
        fused     = _rrf([bm25_hits, vec_hits])[:top_k]

        log.info(
            f"[search] bm25={len(bm25_hits)} vec={len(vec_hits)} "
            f"fused_top3: {[(round(c.score,4), c.meta.get('source_id','?')) for c in fused[:3]]}"
        )

        if not fused:
            return Result(query, [], "no_match")

        self._cache.put_l1(query, fused[0])
        await self._cache.put_l2(query, fused[0])
        return Result(query, fused, "fusion")

/home/duoc/app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-29 12:35:08.979893672 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [3]:
"""DocumentLoaderService v2.1 — Tải và làm sạch tài liệu cho RAG ZERO v2.1.
Không phụ thuộc llama-index. Trả về plain dict để pipeline RAG xử lý.
"""

from __future__ import annotations

import os
import re
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Iterator
from time import sleep

import trafilatura

logger = logging.getLogger(__name__)


@dataclass
class LoadedDocument:
    """Thay thế llama_index.Document — chỉ cần text + metadata."""
    text: str
    metadata: Dict = field(default_factory=dict)


class DocumentLoaderService:
    """
    Dịch vụ tải tài liệu chuẩn hóa cho RAG ZERO v2.1.
    Không phụ thuộc llama-index. Xử lý tiền kỳ dữ liệu thô.
    """

    SUPPORTED_EXTENSIONS: List[str] = [".pdf", ".docx", ".txt", ".csv", ".xlsx", ".md"]
    MAX_FILE_SIZE_MB: int = 50

    _EXCESS_NEWLINE_PATTERN: re.Pattern = re.compile(r"\n{3,}")

    def __init__(self, request_timeout_seconds: int = 15, max_retries: int = 3) -> None:
        self._timeout = request_timeout_seconds
        self._max_retries = max_retries

    def _clean_text(self, raw_text: str) -> str:
        """Làm sạch văn bản thô, giữ ranh giới đoạn văn."""
        if not raw_text:
            return ""
        lines = [line.strip() for line in raw_text.split("\n")]
        joined = "\n".join(lines)
        cleaned = self._EXCESS_NEWLINE_PATTERN.sub("\n\n", joined)
        return cleaned.strip()

    # ── Web Loading ───────────────────────────────────────────────────────────

    def load_web(self, url: str) -> LoadedDocument:
        """Cào web với retry + backoff."""
        last_exc = None
        for attempt in range(self._max_retries):
            try:
                downloaded = trafilatura.fetch_url(url, timeout=self._timeout)
                if downloaded:
                    break
            except Exception as exc:
                last_exc = exc
                if attempt < self._max_retries - 1:
                    sleep(2 ** attempt)  # exponential backoff: 1s, 2s, 4s
                continue
        else:
            raise ValueError(f"Không thể tải URL sau {self._max_retries} lần thử: {url}") from last_exc

        extracted = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=True,
            no_fallback=False,
            target_language="vi",  # ← tối ưu: chỉ extract tiếng Việt
        )

        if not extracted:
            raise ValueError(f"Không trích xuất được nội dung từ URL: {url}")

        return LoadedDocument(
            text=self._clean_text(extracted),
            metadata={
                "source_url": url,
                "document_type": "web_page",
                "storage_layer": "external_crawl",
            },
        )

    # ── File Loading ──────────────────────────────────────────────────────────

    def _check_file_size(self, file_path: Path) -> None:
        """Giới hạn file size để tránh OOM."""
        size_mb = file_path.stat().st_size / (1024 * 1024)
        if size_mb > self.MAX_FILE_SIZE_MB:
            raise ValueError(
                f"File {file_path.name} ({size_mb:.1f}MB) vượt quá giới hạn {self.MAX_FILE_SIZE_MB}MB"
            )

    def _read_pdf(self, path: Path) -> str:
        """Đọc PDF bằng pypdf — nhẹ, không cần llama-index."""
        try:
            from pypdf import PdfReader
        except ImportError:
            raise ImportError("Cài đặt pypdf: pip install pypdf")
        
        reader = PdfReader(str(path))
        texts = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
                texts.append(text)
        return "\n".join(texts)

    def _read_docx(self, path: Path) -> str:
        """Đọc Word bằng python-docx."""
        try:
            from docx import Document
        except ImportError:
            raise ImportError("Cài đặt python-docx: pip install python-docx")
        
        doc = Document(str(path))
        return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

    def _read_xlsx(self, path: Path) -> str:
        """Đọc Excel bằng openpyxl."""
        try:
            from openpyxl import load_workbook
        except ImportError:
            raise ImportError("Cài đặt openpyxl: pip install openpyxl")
        
        wb = load_workbook(str(path), data_only=True)
        texts = []
        for sheet in wb.worksheets:
            for row in sheet.iter_rows(values_only=True):
                row_text = " | ".join(str(cell) for cell in row if cell is not None)
                if row_text.strip():
                    texts.append(row_text)
        return "\n".join(texts)

    def _read_txt(self, path: Path) -> str:
        """Đọc text file với encoding detection đơn giản."""
        for encoding in ["utf-8", "utf-16", "cp1252", "iso-8859-1"]:
            try:
                return path.read_text(encoding=encoding)
            except UnicodeDecodeError:
                continue
        raise ValueError(f"Không thể decode file: {path}")

    def _read_file_by_type(self, path: Path) -> str:
        """Router đọc file theo định dạng."""
        suffix = path.suffix.lower()
        readers = {
            ".pdf": self._read_pdf,
            ".docx": self._read_docx,
            ".xlsx": self._read_xlsx,
            ".txt": self._read_txt,
            ".csv": self._read_txt,
            ".md": self._read_txt,
        }
        reader = readers.get(suffix)
        if not reader:
            raise ValueError(f"Không có reader cho định dạng: {suffix}")
        return reader(path)

    def load_file(self, path: str) -> List[LoadedDocument]:
        """Đọc file đơn lẻ với kiểm tra nghiêm ngặt."""
        file_path = Path(path)

        if not file_path.exists():
            raise FileNotFoundError(f"Không tìm thấy file: {path}")

        if file_path.suffix.lower() not in self.SUPPORTED_EXTENSIONS:
            raise ValueError(
                f"Định dạng '{file_path.suffix}' không được hỗ trợ. "
                f"Chấp nhận: {', '.join(self.SUPPORTED_EXTENSIONS)}"
            )

        self._check_file_size(file_path)

        raw_text = self._read_file_by_type(file_path)
        cleaned = self._clean_text(raw_text)

        if not cleaned:
            raise ValueError(f"Không trích xuất được nội dung từ: {file_path.name}")

        return [LoadedDocument(
            text=cleaned,
            metadata={
                "file_name": file_path.name,
                "file_extension": file_path.suffix.lower().lstrip("."),
                "file_size_bytes": file_path.stat().st_size,
                "absolute_path": str(file_path.resolve()),
                "document_type": "file",
            },
        )]

    # ── Directory Loading ─────────────────────────────────────────────────────

    def _should_skip(self, path: Path) -> bool:
        """Kiểm tra path có nên bỏ qua không (hidden, venv, etc.)."""
        name = path.name
        skip_patterns = (".", "~", "node_modules", ".venv", "__pycache__", ".git")
        return any(pattern in str(path) for pattern in skip_patterns)

    def _iter_files(self, dir_path: Path) -> Iterator[Path]:
        for file_path in dir_path.rglob("*"):
            if file_path.is_file() and file_path.suffix.lower() in self.SUPPORTED_EXTENSIONS:
                yield file_path

    def load_directory(self, path: str) -> List[LoadedDocument]:
        """Quét đệ quy thư mục, stream từng file."""
        dir_path = Path(path)
        if not dir_path.is_dir():
            raise NotADirectoryError(f"Không phải thư mục: {path}")

        processed_docs: List[LoadedDocument] = []
        
        for file_path in self._iter_files(dir_path):
            try:
                self._check_file_size(file_path)
                raw_text = self._read_file_by_type(file_path)
                cleaned = self._clean_text(raw_text)
                
                if cleaned:
                    processed_docs.append(LoadedDocument(
                        text=cleaned,
                        metadata={
                            "file_name": file_path.name,
                            "file_extension": file_path.suffix.lower().lstrip("."),
                            "file_size_bytes": file_path.stat().st_size,
                            "absolute_path": str(file_path.resolve()),
                            "document_type": "file",
                        },
                    ))
            except Exception as exc:
                logger.warning("Bỏ qua file lỗi %s: %s", file_path, exc)
                continue

        logger.info("[Loader] Đã nạp %s tài liệu từ %s", len(processed_docs), path)
        return processed_docs

In [1]:
import sys
import os

os.chdir("/home/duoc/app")        # chuyển về thư mục project
sys.path.insert(0, "/home/duoc/app")  # đảm bảo Python tìm thấy config/
import asyncio
import shutil
import json
import logging
from pathlib import Path
from agent_os.rag.rag_service import RAG



# Cấu hình logging để dễ theo dõi
logging.basicConfig(level=logging.WARNING)

async def run_evaluation():
    PERSIST_DIR = Path("./rag_storage")
    
    # 1. Cleanup an toàn
    if PERSIST_DIR.exists():
        try:
            shutil.rmtree(PERSIST_DIR)
        except Exception as e:
            print(f"Warning: Could not clear storage: {e}")

    rag = RAG()

    # 2. Dữ liệu Test
    docs = [
        ("Quy định nghỉ phép năm: Mỗi năm nhân viên chính thức được hưởng 12 ngày phép có lương. Nhân viên thử việc chưa được tính phép năm. Phép năm không dùng hết được chuyển tối đa 3 ngày sang năm sau, phần còn lại mất.", {"source_id": "hr_001", "category": "hr"}),
        ("Quy trình xin nghỉ phép: Nhân viên phải gửi đơn xin phép qua hệ thống HRM trước ít nhất 3 ngày làm việc. Nghỉ đột xuất do bệnh cần thông báo cho quản lý trực tiếp trước 8h sáng ngày nghỉ và nộp giấy khám bệnh trong vòng 2 ngày.", {"source_id": "hr_002", "category": "hr"}),
        ("Nghỉ thai sản: Lao động nữ được nghỉ thai sản 6 tháng theo quy định pháp luật. Trong thời gian nghỉ thai sản được hưởng chế độ bảo hiểm xã hội. Sau khi hết thai sản, nhân viên được đảm bảo quay lại vị trí cũ hoặc vị trí tương đương.", {"source_id": "hr_003", "category": "hr"}),
        ("Chính sách thưởng tháng 13: Thưởng tháng 13 được chi trả vào kỳ lương cuối tháng 1 dương lịch. Điều kiện: nhân viên làm việc đủ 12 tháng trong năm được hưởng 100%. Làm việc từ 6-11 tháng được hưởng theo tỷ lệ. Dưới 6 tháng không được thưởng tháng 13.", {"source_id": "hr_004", "category": "hr"}),
        ("Cơ cấu lương: Lương gross bao gồm lương cơ bản, phụ cấp chức vụ, phụ cấp xăng xe và phụ cấp điện thoại. Lương net = lương gross - BHXH (8%) - BHYT (1.5%) - BHTN (1%) - thuế TNCN. Lương được thanh toán vào ngày 10 hàng tháng.", {"source_id": "hr_005", "category": "hr"}),
        ("Xét tăng lương: Công ty xét tăng lương định kỳ 1 năm/lần vào tháng 4. Mức tăng dựa trên kết quả đánh giá KPI và thâm niên. Nhân viên đạt KPI xuất sắc được xét tăng lương ngoài chu kỳ. Mức tăng tối thiểu là 5%, tối đa 20%.", {"source_id": "hr_006", "category": "hr"}),
        ("Bảo mật source code: Tuyệt đối không được public mã nguồn của công ty lên github cá nhân hoặc bất kỳ nền tảng công khai nào. Mọi repository phải để private. Vi phạm lần đầu bị cảnh cáo, lần hai bị chấm dứt hợp đồng.", {"source_id": "it_001", "category": "security"}),
        ("Chính sách mật khẩu: Mật khẩu phải có tối thiểu 12 ký tự, bao gồm chữ hoa, chữ thường, số và ký tự đặc biệt. Không được dùng lại mật khẩu cũ trong 5 lần gần nhất. Phải đổi mật khẩu mỗi 90 ngày. Không được chia sẻ mật khẩu với đồng nghiệp dù bất kỳ lý do gì.", {"source_id": "it_002", "category": "security"}),
        ("Sử dụng thiết bị cá nhân (BYOD): Nhân viên được phép dùng máy tính cá nhân làm việc nhưng phải cài đặt phần mềm diệt virus được công ty phê duyệt. Không được lưu dữ liệu khách hàng trên thiết bị cá nhân. Thiết bị cá nhân không được kết nối vào mạng nội bộ (intranet) của công ty.", {"source_id": "it_003", "category": "security"}),
        ("Quy định email công ty: Email công ty chỉ được dùng cho mục đích công việc. Không được đăng ký tài khoản mạng xã hội hoặc dịch vụ cá nhân bằng email công ty. Không được forward email nội bộ ra ngoài khi chưa được phê duyệt.", {"source_id": "it_004", "category": "security"}),
        ("Quy trình onboarding: Nhân viên mới cần hoàn thành các bước: ký hợp đồng lao động, nhận thiết bị làm việc, tạo tài khoản hệ thống, tham gia khóa đào tạo nội quy 2 ngày. Thời gian thử việc là 2 tháng đối với nhân viên và 3 tháng đối với quản lý.", {"source_id": "hr_007", "category": "hr"}),
        ("Quy trình nghỉ việc: Nhân viên muốn nghỉ việc phải nộp đơn trước 30 ngày (quản lý cấp trung: 45 ngày). Phải bàn giao công việc đầy đủ, hoàn trả thiết bị công ty, xóa dữ liệu công ty trên thiết bị cá nhân. Thanh toán lương và các khoản còn lại trong vòng 7 ngày làm việc sau ngày cuối.", {"source_id": "hr_008", "category": "hr"}),
        ("Chính sách đào tạo: Công ty hỗ trợ 100% học phí cho các khóa học liên quan đến công việc, tối đa 10 triệu/năm/người. Nhân viên được cấp 3 ngày học có lương/năm. Sau khi hoàn thành khóa học phải cam kết làm việc thêm ít nhất 12 tháng, nếu nghỉ sớm phải hoàn lại học phí theo tỷ lệ.", {"source_id": "hr_009", "category": "hr"}),
        ("Chính sách remote work: Nhân viên được làm việc từ xa tối đa 2 ngày/tuần sau khi hết thử việc. Phải online trên Slack trong giờ hành chính 8h-17h30. Các buổi họp quan trọng và ngày thứ Hai bắt buộc có mặt tại văn phòng. Cần báo cáo tiến độ công việc hàng ngày qua hệ thống.", {"source_id": "hr_010", "category": "hr"}),
        ("Phụ cấp làm việc từ xa: Nhân viên làm remote được hỗ trợ 500.000 VNĐ/tháng tiền điện và internet. Công ty không hỗ trợ mua thiết bị cho remote work. Phụ cấp remote chỉ áp dụng cho nhân viên chính thức, không áp dụng cho nhân viên thử việc hoặc freelancer.", {"source_id": "hr_011", "category": "hr"}),
    ]

    # 3. Ingest
    print("=== STARTING INGEST ===")
    for text, meta in docs:
        await rag.add(text, **meta)

    # 4. Evaluation Loop
    queries = [
        ("Nghỉ thai sản bao lâu?", "hr_003"),
        ("Mật khẩu cần gì?", "it_002"),
        ("Có được dùng email công ty đăng ký Shopee không?", "it_004"), # Semantic check
        ("Nghỉ việc có phải trả thiết bị không?", "hr_008"),
        ("Remote work mỗi tuần mấy ngày?", "hr_010"),
    ]

    eval_log = []
    correct = 0
    
    print("\n=== STARTING EVALUATION ===")
    for q, expected in queries:
        res = await rag.search(q, top_k=2)
        # Kiểm tra xem expected có trong top_k không (Recall@2)
        found_ids = [c.meta.get("source_id") for c in res.chunks]
        is_hit = expected in found_ids
        
        if is_hit: correct += 1
        
        eval_log.append({
            "query": q,
            "expected": expected,
            "got": found_ids,
            "source": res.source,
            "is_hit": is_hit
        })
        
        mark = "✅" if is_hit else "❌"
        print(f"{mark} Expected: {expected} | Got: {found_ids} | Query: {q}")

    # 5. Output
    print(f"\nFinal Accuracy (Recall@2): {correct/len(queries)*100:.1f}%")
    with open("eval_results.json", "w", encoding="utf-8") as f:
        json.dump(eval_log, f, indent=4, ensure_ascii=False)
        print("Results saved to eval_results.json")

if __name__ == "__main__":
    await run_evaluation()

/home/duoc/app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-29 19:12:04.975973448 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
INFO:rag:[add] f660f103d46b2553 → 1 chunk(s)
INFO:rag:[add] fdd8bd71d527c53c → 1 chunk(s)
INFO:rag:[add] ffda05bb8ecc1abb → 1 chunk(s)
INFO:rag:[add] 5ea838d082f8aa1d → 1 chunk(s)
INFO:rag:[add] 53e8e29e12e25d32 → 1 chunk(s)
INFO:rag:[add] d9ebc6d339696513 → 1 chunk(s)
INFO:rag:[add] ad0fe906b088e0c5 → 1 chunk(s)
INFO:rag:[add] e55a7ae589e463bb → 1 chunk(s)


=== STARTING INGEST ===


INFO:rag:[add] 25eab13c04c8c3f6 → 1 chunk(s)
INFO:rag:[add] 89d615b115d83317 → 1 chunk(s)
INFO:rag:[add] ae10bf9cc3307557 → 1 chunk(s)
INFO:rag:[add] 8ddcd56cf36d0cb5 → 1 chunk(s)
INFO:rag:[add] 8cb5d2efada57fae → 1 chunk(s)
INFO:rag:[add] 771d8315c1d77ae1 → 1 chunk(s)
INFO:rag:[add] e76862bf87c090c4 → 1 chunk(s)
INFO:rag:[search] bm25=1 vec=6 fused_top3: [(0.0328, 'hr_003')]
INFO:rag:[search] bm25=1 vec=6 fused_top3: [(0.0328, 'it_002')]
INFO:rag:[search] bm25=2 vec=6 fused_top3: [(0.0328, 'it_004')]
INFO:rag:[search] bm25=6 vec=6 fused_top3: [(0.0328, 'hr_008'), (0.0315, 'hr_001')]
INFO:rag:[search] bm25=4 vec=6 fused_top3: [(0.0328, 'hr_011'), (0.032, 'hr_010')]



=== STARTING EVALUATION ===
✅ Expected: hr_003 | Got: ['hr_003'] | Query: Nghỉ thai sản bao lâu?
✅ Expected: it_002 | Got: ['it_002'] | Query: Mật khẩu cần gì?
✅ Expected: it_004 | Got: ['it_004'] | Query: Có được dùng email công ty đăng ký Shopee không?
✅ Expected: hr_008 | Got: ['hr_008', 'hr_001'] | Query: Nghỉ việc có phải trả thiết bị không?
✅ Expected: hr_010 | Got: ['hr_011', 'hr_010'] | Query: Remote work mỗi tuần mấy ngày?

Final Accuracy (Recall@2): 100.0%
Results saved to eval_results.json


In [4]:
import sys
import os

os.chdir("/home/duoc/app")        # chuyển về thư mục project
sys.path.insert(0, "/home/duoc/app")  # đảm bảo Python tìm thấy config/

from pathlib import Path
from agent_os.rag.document_loader_service import DocumentLoader
# 1. Thiết lập thư mục test
test_dir = "test_data"
os.makedirs(test_dir, exist_ok=True)

# 2. Tạo một file mẫu
test_file = Path(test_dir) / "test_doc.txt"
with open(test_file, "w", encoding="utf-8") as f:
    f.write("Dòng 1: Chào mừng tới RAG ZERO.\n\n\nDòng 2: Đây là tài liệu test sạch.\n\n\n\nDòng 3: Hệ thống hoạt động tốt.")

# 3. Test Service
loader = DocumentLoader()

print("--- Bắt đầu test ---")

# Test load file
try:
    docs = loader.load_file(str(test_file))
    print(f"Đã đọc thành công {len(docs)} file.")
    print(f"Metadata file: {docs[0].metadata['file_name']}")
    print(f"Text sau khi clean (đã ép 4 dòng trống xuống 2):")
    print(repr(docs[0].text))
except Exception as e:
    print(f"Lỗi khi load_file: {e}")

# Test load directory
try:
    all_docs = loader.load_directory(test_dir)
    print(f"\nĐã quét thư mục, tìm thấy {len(all_docs)} tài liệu.")
except Exception as e:
    print(f"Lỗi khi load_directory: {e}")

# Dọn dẹp sau khi test (tùy chọn)
# import shutil
# shutil.rmtree(test_dir)

INFO:agent_os.rag.document_loader_service:[Loader] Đã nạp 1 tài liệu từ test_data


--- Bắt đầu test ---
Đã đọc thành công 1 file.
Metadata file: test_doc.txt
Text sau khi clean (đã ép 4 dòng trống xuống 2):
'Dòng 1: Chào mừng tới RAG ZERO.\n\nDòng 2: Đây là tài liệu test sạch.\n\nDòng 3: Hệ thống hoạt động tốt.'

Đã quét thư mục, tìm thấy 1 tài liệu.
